In [ ]:
# ! pip install transformers
# ! pip install sentencepiese

In [3]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

ko_text = "이것은 m2m모델로 만든 다국어 번역기 입니다."
chinese_text = "生活就像一盒巧克力。"

model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

# translate korean to english
tokenizer.src_lang = "ko"
encoded_hi = tokenizer(ko_text, return_tensors="pt")
generated_tokens = model.generate(**encoded_hi, forced_bos_token_id=tokenizer.get_lang_id("en"))
result1 = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print(result1)
# => "La vie est comme une boîte de chocolat."

# translate Chinese to English
tokenizer.src_lang = "ko"
encoded_zh = tokenizer(ko_text, return_tensors="pt")
generated_tokens = model.generate(**encoded_zh, forced_bos_token_id=tokenizer.get_lang_id("ja"))
result2 = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print(result2)
# => "Life is like a box of chocolate."

['This is a multi-language translator made with the m2m model.']
['これはm2mモデルで作られた多言語翻訳機です。']


In [4]:
import gradio as gr
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

# 모델 및 토크나이저 로드
model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

# 지원 언어 리스트
languages = {
    "Korean": "ko",
    "English": "en",
    "Japanese": "ja",
    "Chinese (Simplified)": "zh",
    "French": "fr",
    "German": "de",
    "Spanish": "es",
    "Russian": "ru"
}

# 번역 함수
def translate(text, src_lang_name, tgt_lang_name):
    src_lang = languages[src_lang_name]
    tgt_lang = languages[tgt_lang_name]

    tokenizer.src_lang = src_lang
    encoded = tokenizer(text, return_tensors="pt")
    generated_tokens = model.generate(**encoded, forced_bos_token_id=tokenizer.get_lang_id(tgt_lang))
    translated = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    return translated[0]

# Gradio 인터페이스 구성
with gr.Blocks() as demo:
    gr.Markdown("## 🌍 다국어 번역기 (M2M100 기반)")
    
    with gr.Row():
        src_lang = gr.Dropdown(choices=list(languages.keys()), value="Korean", label="원본 언어")
        tgt_lang = gr.Dropdown(choices=list(languages.keys()), value="English", label="번역 언어")

    with gr.Row():
        input_text = gr.Textbox(lines=6, label="입력 텍스트")
        output_text = gr.Textbox(lines=6, label="번역 결과", interactive=False)

    translate_button = gr.Button("번역하기")

    translate_button.click(fn=translate, inputs=[input_text, src_lang, tgt_lang], outputs=output_text)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [5]:
demo.close

<bound method Blocks.close of Gradio Blocks instance: 1 backend functions
-------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x0000014A32E408D0>
 |-<gradio.components.dropdown.Dropdown object at 0x0000014AA6E24ED0>
 |-<gradio.components.dropdown.Dropdown object at 0x0000014AA76DD190>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x0000014A32EE54D0>>

In [10]:
import gradio as gr
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from datasets import load_dataset
import torch
import soundfile as sf
import os

# M2M100 모델 로드
model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

# TTS 모델 로드
processor_tts = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model_tts = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

# 스피커 임베딩 로드
embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
speaker_embeddings = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)

# 지원 언어 목록
languages = {
    "Korean": "ko",
    "English": "en",
    "Japanese": "ja",
    "Chinese (Simplified)": "zh",
    "French": "fr",
    "German": "de",
    "Spanish": "es",
    "Russian": "ru"
}

# 번역 및 영어 TTS
def translate_and_tts(text, src_lang_name, tgt_lang_name):
    src_lang = languages[src_lang_name]
    tgt_lang = languages[tgt_lang_name]

    # 번역
    tokenizer.src_lang = src_lang
    encoded = tokenizer(text, return_tensors="pt")
    generated_tokens = model.generate(**encoded, forced_bos_token_id=tokenizer.get_lang_id(tgt_lang))
    translated = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

    audio_path = None
    if tgt_lang == "en":
        # 영어일 경우 TTS 수행
        inputs_tts = processor_tts(text=translated, return_tensors="pt")
        speech = model_tts.generate_speech(inputs_tts["input_ids"], speaker_embeddings, vocoder=vocoder)
        audio_path = "speech.wav"
        sf.write(audio_path, speech.numpy(), samplerate=16000)

    return (translated, audio_path)

# Gradio 인터페이스
with gr.Blocks() as demo:
    gr.Markdown("## 🌍 다국어 번역기 + 🗣️ 영어 음성 읽기 (M2M100 + SpeechT5)")

    with gr.Row():
        src_lang = gr.Dropdown(choices=list(languages.keys()), value="Korean", label="원본 언어")
        tgt_lang = gr.Dropdown(choices=list(languages.keys()), value="English", label="번역 언어")

    with gr.Row():
        input_text = gr.Textbox(lines=6, label="입력 텍스트")
        output_text = gr.Textbox(lines=6, label="번역 결과", interactive=False)

    audio_output = gr.Audio(label="영어 음성", type="filepath")
    translate_button = gr.Button("번역하기")

    translate_button.click(fn=translate_and_tts,
                           inputs=[input_text, src_lang, tgt_lang],
                           outputs=[output_text, audio_output])

demo.launch()

C:\Users\Admin\miniforge3\envs\ai_serving\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--microsoft--speecht5_tts. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\Admin\miniforge3\envs\ai_serving\Lib\site-packages\huggingface_hub\file_download.py:143: UserWa

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [9]:
! pip install gradio transformers datasets soundfile torch

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 12.2 MB/s eta 0:00:00

   ---------------------------------------- 0/3 [pycparser]
   ---------------------------------------- 0/3 [pycparser]
   ------------- -------------------------- 1/3 [cffi]
   ------------- -------------------------- 1/3 [cffi]
   ------------- -------------------------- 1/3 [cffi]
   -------------------------- ------------- 2/3 [soundfile]
   ---------------------------------------- 3/3 [soundfile]

